In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tkinter as tk

from tkinter import messagebox
from PIL import Image, ImageTk

In [2]:
df = pd.read_csv('dataset_penyisihan_bdc_2024.csv', sep=';')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5000 non-null   str  
 1   label   5000 non-null   str  
dtypes: str(2)
memory usage: 78.3 KB


In [4]:
df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,RT Anies dapat tepuk tangan meriah saat jadi R...,Politik
2,@CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...,Demografi
3,RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [5]:
df[df.duplicated()]

,text,label
57,RT Abah Anies ingin mengangkat martabat petani...,Sosial Budaya
104,"RT Anak Muda Indonesia, the future of this nat...",Ideologi
145,"RT Pupuk bersubsidi langka, Tim Prabowo Gibran...",Ekonomi
146,RT Abah Anies ingin mengangkat martabat petani...,Sosial Budaya
189,RT Abah Anies ingin mengangkat martabat petani...,Sosial Budaya
...,...,...
4917,"RT Ini kejam, warga disitu sudah bersedia diba...",Sosial Budaya
4933,RT Karena program pak anies yg ini. Bapak gw j...,Sumber Daya Alam
4955,RT Ekonom: Program Makan Siang dan Susu Gratis...,Ekonomi
4971,"RT menjelang tengah malam ini, aku mau ucapkan...",Ideologi


In [6]:
df.duplicated().sum()

np.int64(381)

In [7]:
df = df.drop_duplicates()

In [8]:
df['label'].value_counts()

label
Politik                    2972
Sosial Budaya               425
Ideologi                    343
Pertahanan dan Keamanan     331
Ekonomi                     310
Sumber Daya Alam            157
Demografi                    61
Geografi                     20
Name: count, dtype: int64

In [9]:
df['label'] = df['label'].map({'Politik':0, 'Sosial Budaya':1, 'Ideologi':2, 'Pertahanan dan Keamanan':3, 'Ekonomi':4, 'Sumber Daya Alam':5, 'Demografi':6, 'Geografi':7})

In [10]:
X = df['text']
y = df['label']

In [11]:
def split_data(X, y, size=0.8):
    np.random.seed(42)

    x_train, x_test = [], []
    y_train, y_test = [], []

    for classes in np.unique(y):
        class_idx = np.where(y == classes)[0]

        idx = np.random.permutation(class_idx)
        split = int(len(idx) * size)

        x_train.append(X.iloc[idx[:split]])
        x_test.append(X.iloc[idx[split:]])
        y_train.append(y.iloc[idx[:split]])
        y_test.append(y.iloc[idx[split:]])

    X_train  = pd.concat(x_train).reset_index(drop=True)
    X_test  = pd.concat(x_test).reset_index(drop=True)
    y_train  = pd.concat(y_train).reset_index(drop=True)
    y_test  = pd.concat(y_test).reset_index(drop=True)

    return X_train, X_test, y_train, y_test

In [12]:
X_train, X_test, y_train, y_test = split_data(X, y)

for var in [X_train, X_test, y_train, y_test]:
    print(var.shape)

(3692,)
(927,)
(3692,)
(927,)


In [13]:
def prior(y, classes):
    n_sample = len(y)
    prior = np.zeros(len(classes))

    for idx, c in enumerate(classes):
        prios[idx] = np.sum(y == c) / n_sample

        return prior

In [14]:
def likehood(X, y, classes, alpha=1.0):
    n_feature = X.shape[1]
    n_classes = len(classes)

    likelihood = np.zeros((n_classes, n_feature))

    for idx, c in enumerate(classes):
        X_c = X[y == c]
        total_count = np.sum(X_c)
        feature_count = np.sum(X_c, axis=0)
        likelihood[idx, :] = (feature_count + alpha) / (total_count + alpha & n_feature)
        
        return likelihood

In [15]:
class naivebayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.prior = None
        self.likelihood = None
        self.log_prior = None
        self.log_likelihood = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        n_classses = len(self.classes)
        n_feature = X.shape[1]

        self.prior = np.zeros(n_classses)
        for idx, c in enumerate(self.classes):
            self.prior[idx] = np.sum(y == c) / (len(y) + 1e-10)

        self.likelihood = np.zeros((n_classses, n_feature))

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]

            if len(X_c) == 0:
                self.likelihood[idx] = np.ones(n_feature) / n_feature
                continue

            total_count = np.sum(c)
            feature_count = np.sum(X_c, axis=0)

            numerator = feature_count + self.alpha
            denominator = total_count + (self.alpha * n_feature)

            if denominator == 0:
                self.likelihood[idx] = np.ones(n_feature) / n_feature
            else:
                self.likelihood[idx] = numerator / denominator

            self.log_prior = np.log(self.prior)
            self.likelihood = np.clip(self.likelihood, 1e-10, 1.0)
            self.log_likelihood = np.log(self.likelihood)

            return self

    def predict_single(self, x):
        posterior = np.zeros(len(self.classes))
        for idx in range(len(self.classes)):
            posterior[idx] = self.log_prior[idx]
            posterior[idx] += np.sum(x * self.log_likelihood[idx])
    
        return self.classes[np.argmax(posterior)]

    def predict(self, X):
        prediction = np.array([self.predict_single(x) for x in X])
        
        return prediction

    def predict_probs(self, X):
        probs = np.zeros((X.shape[0], len(self.classes)))

        for i, x in enumerate(X):
            log_posterios = np.zeros(len(self.classes))

            for idx in range(len(self.classes)):
                log_posterios[idx] = self.log_prior[idx]
                log_posterios[idx] += np.sum(x * self.log_likelihood[idx])

            log_posterios -= np.max(log_posterios)
            exp_posterior = np.exp(log_posterios)
            probs[i] =  exp_posterior / np.sum(exp_posterior)

        return probs

In [16]:
import numpy as np

# Contoh data: Document classification (spam detection)
# X: matrix TF-IDF atau count vector (n_samples, n_features)
# y: label class (n_samples,)

# Data dummy
X = np.array([
    [2, 1, 0, 3, 0],  # "buy cheap pills now" -> spam
    [0, 0, 2, 1, 1],  # "meeting tomorrow schedule" -> ham
    [3, 2, 0, 2, 0],  # "cheap buy now cheap" -> spam
    [0, 1, 3, 0, 2],  # "schedule meeting today important" -> ham
    [1, 0, 1, 2, 1],  # "buy schedule now" -> ham
])

y = np.array([1, 0, 1, 0, 0])  # 1=spam, 0=ham

print("Data shape:", X.shape)
print("Classes:", np.unique(y))

Data shape: (5, 5)
Classes: [0 1]


In [17]:
# Training
model = naivebayes(alpha=1.0)
model.fit(X, y)

print("=" * 50)
print("MODEL TRAINED")
print("=" * 50)
print("Classes:", model.classes)
print("Priors:", model.prior)
print("\nLog Priors:", model.log_prior)
print("\nLikelihood (P(feature|class)):")
for i, c in enumerate(model.classes):
    print(f"  Class {c}: {model.likelihood[i]}")

# Test dengan data baru
X_test = np.array([
    [2, 2, 0, 3, 0],  # Mirip spam: banyak "cheap", "buy", "now"
    [0, 0, 2, 0, 2],  # Mirip ham: "meeting", "schedule"
])

print("\n" + "=" * 50)
print("PREDICTION")
print("=" * 50)

predictions = model.predict(X_test)
probabilities = model.predict_probs(X_test)

for i, (x, pred, prob) in enumerate(zip(X_test, predictions, probabilities)):
    print(f"\nTest sample {i}: {x}")
    print(f"  Predicted class: {pred}")
    print(f"  Probabilities: {dict(zip(model.classes, prob))}")

# Manual calculation verification
print("\n" + "=" * 50)
print("MANUAL VERIFICATION (Sample 0)")
print("=" * 50)

x = X_test[0]
print(f"Sample: {x}")

for idx, c in enumerate(model.classes):
    log_prior = model.log_prior[idx]
    log_lik = np.sum(x * model.log_likelihood[idx])
    log_post = log_prior + log_lik
    
    print(f"\nClass {c}:")
    print(f"  log(P({c})) = {log_prior:.4f}")
    print(f"  sum(x * log(P(x|{c}))) = {log_lik:.4f}")
    print(f"  log_posterior = {log_post:.4f}")

MODEL TRAINED
Classes: [0 1]
Priors: [0.6 0.4]

Log Priors: [-0.51082562 -0.91629073]

Likelihood (P(feature|class)):
  Class 0: [0.4 0.4 1.  0.8 1. ]
  Class 1: [1.e-10 1.e-10 1.e-10 1.e-10 1.e-10]

PREDICTION

Test sample 0: [2 2 0 3 0]
  Predicted class: 0
  Probabilities: {np.int64(0): np.float64(1.0), np.int64(1): np.float64(5.086263020833219e-69)}

Test sample 1: [0 0 2 0 2]
  Predicted class: 0
  Probabilities: {np.int64(0): np.float64(1.0), np.int64(1): np.float64(6.666666666666685e-41)}

MANUAL VERIFICATION (Sample 0)
Sample: [2 2 0 3 0]

Class 0:
  log(P(0)) = -0.5108
  sum(x * log(P(x|0))) = -4.3346
  log_posterior = -4.8454

Class 1:
  log(P(1)) = -0.9163
  sum(x * log(P(x|1))) = -161.1810
  log_posterior = -162.0972


In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 4619 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    4619 non-null   str  
 1   label   4619 non-null   int64
dtypes: int64(1), str(1)
memory usage: 108.3 KB


In [2]:
def submit():
    input_data = entry.get()
    if not input_data.strip():
        messagebox.showwarning('Warning', 'Masukan Komentar!')
        return
    try:
        labels = {
            0: 'Politik', 1: 'Sosial Budaya', 2: 'Ideologi', 
            3: 'Pertahanan dan Keamanan', 4: 'Ekonomi', 
            5: 'Sumber Daya Alam', 6: 'Demografi', 7: 'Geografi'
        }
    except Exception as e:
        messagebox.showerror('Error', f'Terjadi kesalahan: {e}')

root = tk.Tk()
root.title('Prediksi Kategori Komentar')
root.configure(bg='#131313')
root.geometry('500x500')

frame_tengah = tk.Frame(root, bg='#131313')
frame_tengah.place(relx=0.5, rely=0.4, anchor='center')

# logo
container_logo = tk.Frame(frame_tengah,
                          bg='#131313')
container_logo.pack(pady=10)

img = Image.open('robot.png').resize((80,80))
logo = ImageTk.PhotoImage(img)
label_logo = tk.Label(container_logo,
                      image=logo,
                      bg='#131313')
label_logo.pack()

# Body
title_label = tk.Label(frame_tengah,
                       text='Prediksi Kategori Komentar', 
                       font=('Helvetica', 22, 'bold'), 
                       bg='#131313', 
                       fg='#FFFFFF')
title_label.pack(pady=10)

entry_label = tk.Label(frame_tengah, 
                       text='Masukan Komentar', 
                       font=('Helvetica', 10), 
                       bg='#131313', 
                       fg='#FFFFFF')
entry_label.pack(pady=5)

entry = tk.Entry(frame_tengah, 
                 fg='#FFFFFF', 
                 bg='#222222', 
                 font=('Helvetica', 18), 
                 insertbackground='white', 
                 bd=0)
entry.pack(pady=10, ipady=5)

pred_btn = tk.Button(frame_tengah, 
                     text='Prediksi', 
                     font=('Helvetica', 12, 'bold'), 
                     bg='#1D9BF0', fg='#FFFFFF', 
                     command=submit, 
                     bd=0, 
                     width=15)
pred_btn.pack(pady=20)

# footer
footer_frame = tk.Frame(root, 
                        bg='#131313')
footer_frame.pack(side='bottom', fill='x', pady=10)

line = tk.Frame(footer_frame, 
                height=1, 
                width=400, 
                bg='#444444')
line.pack(pady=5)

label_bottom = tk.Label(footer_frame, 
                        text='Sincerely SMKN 2 Singosari',
                        font=('Helvetica', 8),
                        fg="#666666",
                        bg='#131313')
label_bottom.pack()

root.mainloop()